<a href="https://colab.research.google.com/github/cto-school/mathematics-for-machine-learning/blob/main/06-calculus-and-derivatives/06a_limits_derivatives.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 06a — Limits & Derivatives

This notebook is about **change**. How fast is a function rising or falling at a
single instant? That number is the **derivative**, and almost every machine
learning algorithm finds its answer by following derivatives downhill.

We will build the idea from scratch:

1. A **limit** — sneaking up on a value we cannot reach directly.
2. The **derivative** as the slope of the tangent line / instantaneous rate of
   change.
3. A **numerical derivative** we can compute for *any* function with NumPy.
4. The basic differentiation **rules**, checked numerically.
5. The **linear (Taylor) approximation** — the workhorse of optimization.

Run each cell with **Shift + Enter**. Edit and re-run freely.

## 1. Setup

Pure NumPy for the math, Matplotlib for the pictures. We fix a random seed so
results are reproducible.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)   # reproducible randomness (used later)

## 2. The idea of a limit

The slope of a straight line through two points $(x_1,y_1)$ and $(x_2,y_2)$ is

$$\text{slope} = \frac{y_2 - y_1}{x_2 - x_1} = \frac{\Delta y}{\Delta x}.$$

For a *curve* there is no single slope — it bends. But if we pick two points
that are **very close together**, the line through them (a *secant*) looks almost
like the slope "at" that spot. A **limit** asks: what value does the secant slope
*approach* as the two points squeeze together?

Take $f(x)=x^2$ at $x=2$. The secant slope between $x=2$ and $x=2+h$ is

$$\frac{f(2+h)-f(2)}{h} = \frac{(2+h)^2 - 4}{h} = 4 + h.$$

As $h \to 0$ this clearly approaches $4$. Let's watch it happen numerically.

In [ ]:
def f(x):
    return x**2

x0 = 2.0
for h in [1.0, 0.1, 0.01, 0.001, 0.0001]:
    secant_slope = (f(x0 + h) - f(x0)) / h   # rise over run
    print(f"h = {h:8.4f}   secant slope = {secant_slope:.6f}")

The slope marches toward **4**. We say *"the limit of the secant slope as
$h\to 0$ is 4"*. That limiting slope is the **derivative** of $f$ at $x=2$.

We cannot just plug in $h=0$ (it would be $0/0$), but the limit dodges that
problem: we only ever get *close* to 0.

## 3. The derivative = slope of the tangent line

The **derivative** $f'(x)$ is the limit of the secant slope:

$$f'(x) = \lim_{h \to 0} \frac{f(x+h) - f(x)}{h}.$$

It is the **instantaneous rate of change** of $f$, and geometrically it is the
slope of the **tangent line** — the straight line that just grazes the curve at
$x$, matching both its height and its direction.

Below we draw $f(x)=x^2$ and several secant lines from the point $(2,4)$, each
using a smaller $h$. Watch them rotate toward the tangent (slope 4).

In [ ]:
xs = np.linspace(0, 4, 200)        # 200 points from 0 to 4
plt.plot(xs, f(xs), label="f(x) = x^2", color="black")

x0 = 2.0
for h in [1.5, 1.0, 0.5]:
    slope = (f(x0 + h) - f(x0)) / h           # secant slope
    # line through (x0, f(x0)) with this slope, drawn across the window:
    secant = f(x0) + slope * (xs - x0)
    plt.plot(xs, secant, "--", label=f"secant h={h} (slope {slope:.1f})")

# the true tangent at x0 has slope 4:
tangent = f(x0) + 4.0 * (xs - x0)
plt.plot(xs, tangent, color="red", linewidth=2, label="tangent (slope 4)")

plt.scatter([x0], [f(x0)], color="red", zorder=5)
plt.title("Secant lines approaching the tangent")
plt.xlabel("x"); plt.ylabel("y")
plt.legend(); plt.grid(True)
plt.show()

## 4. A numerical derivative for *any* function

On paper we use rules to get $f'$. But on a computer we can estimate the
derivative of **any** function we can evaluate, using the **central difference**:

$$f'(x) \approx \frac{f(x+h) - f(x-h)}{2h}.$$

It looks left *and* right and averages, which is far more accurate than the
one-sided $\frac{f(x+h)-f(x)}{h}$. A good all-purpose step is $h \approx 10^{-5}$.

In [ ]:
def deriv(f, x, h=1e-5):
    # central difference: (f(x+h) - f(x-h)) / (2h)
    # works elementwise if x is a NumPy array, because f is vectorized
    return (f(x + h) - f(x - h)) / (2 * h)

# check against the known answer f'(x) = 2x for f(x) = x^2
for x in [1.0, 2.0, 3.0]:
    print(f"deriv at x={x}: {deriv(f, x):.6f}   (exact 2x = {2*x})")

## 5. Plot a function and its derivative together

Because `deriv` accepts a whole array, we can evaluate $f'$ on a **grid** and
plot it. Let's use $f(x)=\sin(x)$, whose derivative is famously $\cos(x)$.

In [ ]:
g = np.sin                       # the function
xs = np.linspace(-2*np.pi, 2*np.pi, 400)

y  = g(xs)                       # f
dy = deriv(g, xs)                # numerical f', computed on the whole grid

plt.plot(xs, y,  label="f(x) = sin(x)")
plt.plot(xs, dy, label="f'(x) (numerical)")
plt.plot(xs, np.cos(xs), "k--", label="cos(x) (exact)")
plt.axhline(0, color="gray", linewidth=0.8)
plt.title("A function and its derivative")
plt.xlabel("x"); plt.ylabel("value")
plt.legend(); plt.grid(True)
plt.show()

Notice the relationship: where $f$ has a **peak or valley** (slope zero), $f'$
**crosses zero**. Where $f$ rises, $f'$ is positive; where $f$ falls, $f'$ is
negative. The derivative is a complete "rate-of-change report" on $f$.

## 6. Drawing the tangent line at a point

The tangent line at $x_0$ has slope $f'(x_0)$ and passes through
$(x_0, f(x_0))$:

$$T(x) = f(x_0) + f'(x_0)\,(x - x_0).$$

Let's draw it for $f(x)=\sin(x)$ at $x_0 = 1$.

In [ ]:
x0 = 1.0
slope = deriv(g, x0)             # f'(x0), numerically

xs = np.linspace(-1, 4, 300)
tangent = g(x0) + slope * (xs - x0)   # the tangent-line formula

plt.plot(xs, g(xs), label="f(x) = sin(x)")
plt.plot(xs, tangent, "r", label=f"tangent at x0=1 (slope {slope:.3f})")
plt.scatter([x0], [g(x0)], color="red", zorder=5)
plt.title("Tangent line touches the curve at x0")
plt.xlabel("x"); plt.ylabel("y")
plt.legend(); plt.grid(True)
plt.show()

## 7. The differentiation rules, verified numerically

You will learn these rules in a calculus course; here we *confirm* them by
comparing our numerical `deriv` to the formula each rule predicts.

| Rule | Statement |
|------|-----------|
| Power | $\frac{d}{dx}x^n = n\,x^{n-1}$ |
| Sum | $(u+v)' = u' + v'$ |
| Product | $(uv)' = u'v + uv'$ |
| Chain | $\big(u(v(x))\big)' = u'(v(x))\cdot v'(x)$ |

In [ ]:
x = 1.3   # test point

# --- Power rule: d/dx x^4 = 4 x^3 ---
p = lambda x: x**4
print("power  :", deriv(p, x), " vs ", 4 * x**3)

# --- Sum rule: (x^2 + sin x)' = 2x + cos x ---
s = lambda x: x**2 + np.sin(x)
print("sum    :", deriv(s, x), " vs ", 2*x + np.cos(x))

# --- Product rule: (x^2 * sin x)' = 2x sin x + x^2 cos x ---
pr = lambda x: x**2 * np.sin(x)
print("product:", deriv(pr, x), " vs ", 2*x*np.sin(x) + x**2*np.cos(x))

# --- Chain rule: (sin(x^2))' = cos(x^2) * 2x ---
ch = lambda x: np.sin(x**2)
print("chain  :", deriv(ch, x), " vs ", np.cos(x**2) * 2*x)

Every numerical value matches the rule's prediction to many decimals. The rules
are not magic — they are just bookkeeping for the same limit we computed in
Section 2.

## 8. Linear (Taylor) approximation

Near a point $x$, a smooth function looks almost like its tangent line. This is
the **first-order Taylor / linear approximation**:

$$f(x + \Delta) \approx f(x) + f'(x)\,\Delta.$$

In words: *"to move a little, take the current value plus slope times step."*
This single formula is the seed of gradient-based optimization. Let's see how
good it is for $f(x)=e^x$ around $x=0$ (where $f(0)=1$, $f'(0)=1$).

In [ ]:
h = lambda x: np.exp(x)
x0 = 0.0
slope = deriv(h, x0)

xs = np.linspace(-1.5, 1.5, 300)
approx = h(x0) + slope * (xs - x0)     # f(x0) + f'(x0)*(x - x0)

plt.plot(xs, h(xs), label="f(x) = e^x (exact)")
plt.plot(xs, approx, "r--", label="linear approx at x0=0")
plt.scatter([x0], [h(x0)], color="red", zorder=5)
plt.title("Linear approximation is great near x0, drifts far away")
plt.xlabel("x"); plt.ylabel("y")
plt.legend(); plt.grid(True)
plt.show()

In [ ]:
# Numerically: how close is the approximation for small steps?
x0 = 0.0
for delta in [0.5, 0.1, 0.01]:
    exact  = h(x0 + delta)
    approx = h(x0) + deriv(h, x0) * delta
    print(f"delta={delta:5.2f}   exact={exact:.6f}  approx={approx:.6f}"
          f"  error={abs(exact-approx):.6f}")

The error shrinks fast as the step gets smaller — roughly like $\Delta^2$.
That's why taking *small steps* along the slope is a trustworthy way to move.

---
## ✍️ Exercise 1 (solution included)

Use the central-difference `deriv` to estimate $f'(x)$ for
$f(x) = x^3 - 2x$ at $x = 2$. Compare with the exact derivative
$f'(x) = 3x^2 - 2$.

**Solution:**

In [ ]:
f = lambda x: x**3 - 2*x
x = 2.0
print("numerical:", deriv(f, x))
print("exact    :", 3*x**2 - 2)   # 3*4 - 2 = 10

## ✍️ Exercise 2 (solution included)

For $f(x) = \cos(x)$, plot $f$ and its numerical derivative on $[0, 2\pi]$, and
overlay the exact derivative $-\sin(x)$ as a dashed line to confirm they agree.

**Solution:**

In [ ]:
f = np.cos
xs = np.linspace(0, 2*np.pi, 300)

plt.plot(xs, f(xs),         label="cos(x)")
plt.plot(xs, deriv(f, xs),  label="numerical f'")
plt.plot(xs, -np.sin(xs), "k--", label="-sin(x) (exact)")
plt.title("cos and its derivative")
plt.xlabel("x"); plt.ylabel("value")
plt.legend(); plt.grid(True)
plt.show()

---
## 📝 Homework (no solutions provided)

1. Verify the **quotient rule** numerically: for $f(x)=\dfrac{\sin x}{x}$ at
   $x=1.5$, compare `deriv(f, 1.5)` against
   $\dfrac{x\cos x - \sin x}{x^2}$.
2. Make a table of the central-difference error for $f(x)=e^x$ at $x=0$ using
   $h = 10^{-1}, 10^{-3}, 10^{-5}, 10^{-8}, 10^{-12}$. Why does the error stop
   improving (and even get worse) for very tiny $h$?
3. Plot $f(x)=x^3 - 3x$ together with its numerical derivative on $[-2,2]$, and
   mark with dots the $x$-values where $f'(x)=0$.
4. Using the linear approximation $\sqrt{x+\Delta}\approx\sqrt{x}+
   \frac{1}{2\sqrt{x}}\Delta$ around $x=100$, estimate $\sqrt{101}$ and compare
   with `np.sqrt(101)`.